# License Plate Detection — YOLOv8 (Google Colab)
**CMPS 261 — Machine Learning Project**

Run this on Colab with a **T4 GPU** for fast training.

**Before running:** Upload `license_plate_data.zip` using the cell below.

## 1. Install Dependencies

In [ ]:
!pip install ultralytics -q

## 2. Upload & Extract Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload license_plate_data.zip

In [ ]:
import zipfile, os

with zipfile.ZipFile('license_plate_data.zip', 'r') as z:
    z.extractall('/content/')

print('Extracted:')
print('  Train:', len(os.listdir('/content/data/yolo/images/train')), 'images')
print('  Val  :', len(os.listdir('/content/data/yolo/images/val')),   'images')
print('  Test :', len(os.listdir('/content/data/yolo/images/test')),  'images')

## 3. Fix Dataset YAML Path for Colab

In [ ]:
yaml_content = """
path: /content/data/yolo
train: images/train
val:   images/val
test:  images/test

nc: 1
names: ['licence']
"""

with open('/content/data/yolo/dataset.yaml', 'w') as f:
    f.write(yaml_content)

print('YAML updated for Colab paths.')

## 4. Check GPU

In [ ]:
import torch
print(f'GPU available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name      : {torch.cuda.get_device_name(0)}')

## 5. Train YOLOv8s (Small — better accuracy than Nano, still fast on GPU)

In [ ]:
from ultralytics import YOLO

# YOLOv8s = small model, significantly more accurate than nano (YOLOv8n)
model = YOLO('yolov8s.pt')

results = model.train(
    data      = '/content/data/yolo/dataset.yaml',
    epochs    = 100,
    imgsz     = 800,       # larger image = better detection of small plates
    batch     = 16,
    device    = 0,         # GPU
    optimizer = 'AdamW',   # better optimizer for small datasets
    lr0       = 0.001,     # initial learning rate
    cos_lr    = True,      # cosine LR scheduler for smoother convergence
    augment   = True,      # enable augmentation
    fliplr    = 0.5,       # horizontal flip
    degrees   = 5.0,       # small rotation
    translate = 0.1,       # small translation
    scale     = 0.3,       # scale jitter
    mosaic    = 1.0,       # mosaic augmentation
    project   = '/content/models',
    name      = 'yolov8s_colab',
    exist_ok  = True,
)

## 6. Evaluate on Test Set

In [ ]:
test_metrics = model.val(split='test')
print(f'mAP@0.5      : {test_metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95 : {test_metrics.box.map:.4f}')
print(f'Precision    : {test_metrics.box.mp:.4f}')
print(f'Recall       : {test_metrics.box.mr:.4f}')

## 7. Download the Best Weights

In [ ]:
from google.colab import files
files.download('/content/models/yolov8s_colab/weights/best.pt')